In [0]:
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from requests.exceptions import (
    ChunkedEncodingError,
    ConnectionError,
    ReadTimeout,
    RequestException
)

# ============================================
# Session
# ============================================

session = requests.Session()

retry_strategy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({

    "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/138.0 Safari/537.36"

})

In [0]:
categories = [

    "most-active",
    "trending",
    "gainers",
    "losers",
    "52-week-gainers",
    "52-week-losers",
    "highest-dividend",
    "small-cap-stocks",
    "large-cap-stocks",
    "most-expensive-stocks",
    "oversold-stocks",
    "pink-sheet-stocks",
    "overbought-stocks",
    "all-time-high-stocks",
    "unusual-volume-stocks",
    "highest-beta-stocks",
    "highest-revenue-stocks",
    "highest-cash-stocks",
    "highest-net-income-stocks",
    "highest-profit-per-employee-stocks",
    "highest-revenue-per-employee-stocks",
    "largest-employer-stocks"

]


def get_cell_text(cells, key, default=None):

    cell = cells.get(key)

    if cell is None:
        return default

    return cell.get_text(" ", strip=True)


def extract_price(cells):

    price_cell = cells.get("intradayprice")

    if price_cell is None:
        return None

    span = price_cell.find("span")

    if span:
        return span.get_text(strip=True)

    return None

In [0]:
def fetch_page(url):

    MAX_RETRIES = 5

    for attempt in range(MAX_RETRIES):

        try:

            response = session.get(
                url,
                timeout=60
            )

            response.raise_for_status()

            if len(response.text) < 5000:
                raise Exception(
                    "Incomplete HTML returned."
                )

            return response

        except (

            ChunkedEncodingError,
            ConnectionError,
            ReadTimeout,
            RequestException

        ) as e:

            print(
                f"Retry {attempt+1}/{MAX_RETRIES}"
            )

            print(e)

            time.sleep(
                2 * (attempt + 1)
            )

        except Exception as e:

            print(e)

            time.sleep(
                2 * (attempt + 1)
            )

    return None

In [0]:
# ============================================
# Config for pagination safety
# ============================================

PAGE_SIZE = 100

# Safety cap only - normal categories finish in a handful of pages
# via the "no rows" / "duplicate page" checks below. This just
# guarantees the loop can never spin forever if Yahoo's markup
# changes in a way that breaks those checks.
MAX_PAGES_PER_CATEGORY = 100

SLEEP_BETWEEN_PAGES_SECONDS = 2


def parse_row(row):
    """
    Parse a single <tr> into a dict of {data-testid-cell: <td>}.
    Returns None if the row has no usable cells (e.g. a stray
    header/ad row that slipped through the table selector).
    """

    cells = {}

    for td in row.find_all("td"):

        key = td.get("data-testid-cell")

        if key:
            cells[key] = td

    if not cells:
        return None

    return cells


def parse_table_rows(soup):
    """
    Parse all rows on a page into a list of cell-dicts,
    skipping any row that fails to parse cleanly.
    """

    parsed_rows = []

    for row in soup.select("table tbody tr"):

        try:

            cells = parse_row(row)

            if cells is not None:
                parsed_rows.append(cells)

        except Exception as e:

            print(f"  Skipping malformed row: {e}")

    return parsed_rows


def build_company_record(category, cells):
    """
    Map a parsed row's cells into the output record shape.
    Field mapping is unchanged from the original implementation.
    """

    return {

        "category": category,

        "ticker": get_cell_text(cells, "ticker"),

        "company_name":
            get_cell_text(cells, "companyshortname.raw"),

        "price": extract_price(cells),

        "change":
            get_cell_text(cells, "intradaypricechange"),

        "change_percent":
            get_cell_text(cells, "percentchange"),

        "volume":
            get_cell_text(cells, "dayvolume"),

        "avg_volume_3m":
            get_cell_text(cells, "avgdailyvol3m"),

        "market_cap":
            get_cell_text(cells, "intradaymarketcap"),

        "PE_ratio":
            get_cell_text(cells, "peratio.lasttwelvemonths"),

        "52_wk_change_percent":
            get_cell_text(cells, "fiftytwowkpercentchange"),

        "52_wk_range":
            get_cell_text(cells, "fiftyTwoWeekRange")

    }


def scrape_category(category):
    """
    Scrape every page of a single category until a stop
    condition is hit. Returns the raw parsed rows (list of
    cell-dicts) across all pages - dedup against other
    categories happens at the call site, same as before.
    """

    start = 0

    seen_pages = set()

    category_rows = []

    for _ in range(MAX_PAGES_PER_CATEGORY):

        url = (
            f"https://finance.yahoo.com/markets/stocks/"
            f"{category}/?start={start}&count={PAGE_SIZE}"
        )

        print(f"Scraping : {url}")

        response = fetch_page(url)

        if response is None:

            print("Skipping category.")

            break

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        parsed_rows = parse_table_rows(soup)

        # Match original behavior exactly: rows with no ticker
        # never counted towards page content or dedup signature
        parsed_rows = [
            cells for cells in parsed_rows
            if get_cell_text(cells, "ticker") is not None
        ]

        if not parsed_rows:

            print("No rows found.")

            break

        page_tickers = tuple(
            get_cell_text(cells, "ticker")
            for cells in parsed_rows
        )

        if page_tickers in seen_pages:

            print("Duplicate page detected.")

            break

        seen_pages.add(page_tickers)

        category_rows.extend(parsed_rows)

        start += PAGE_SIZE

        time.sleep(SLEEP_BETWEEN_PAGES_SECONDS)

    else:

        print(
            f"Hit MAX_PAGES_PER_CATEGORY ({MAX_PAGES_PER_CATEGORY}) "
            f"safety cap - stopping category early."
        )

    return category_rows


# ============================================
# Scrape all categories
# ============================================

all_data = []

seen_tickers = set()

for category in categories:

    print(f"\n{'='*60}")
    print(f"Category : {category}")
    print(f"{'='*60}")

    category_rows = scrape_category(category)

    added = 0

    for cells in category_rows:

        ticker = get_cell_text(cells, "ticker")

        if ticker is None:
            continue

        if ticker in seen_tickers:
            continue

        seen_tickers.add(ticker)

        all_data.append(
            build_company_record(category, cells)
        )

        added += 1

    print(f"Added {added} stocks.")


In [0]:
# ============================================
# Pandas DataFrame
# ============================================

market_df = pd.DataFrame(all_data)

print("\n" + "=" * 60)
print(f"Total Unique Companies : {len(market_df)}")
print("=" * 60)

print(market_df.head())


# ============================================
# Spark DataFrame
# ============================================

from pyspark.sql.functions import current_timestamp

spark_df = spark.createDataFrame(
    market_df
)

spark_df = spark_df.withColumn(
    "dwh_loaded_at",
    current_timestamp()
)


# ============================================
# Save To Delta Table
# ============================================

TABLE_NAME = "finance_intelligence_hub.bronze.companies"

spark.sql(f"""
DROP TABLE IF EXISTS {TABLE_NAME}
""")

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_NAME)


print("\n" + "=" * 60)
print("Data saved successfully.")
print(f"Table : {TABLE_NAME}")
print(f"Rows  : {spark_df.count()}")
print("=" * 60)